# F — Peak Lag: Surface vs Air Temperature Timing
Surface temperatures peak earlier than air temperatures due to direct solar heating.
This notebook quantifies the lag between surface and air temperature peaks
across all areas and seasons.

**Saves to** `../figures/F_peak_lag/`

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
sys.path.insert(0, str(NB_DIR))
from config import *
import matplotlib.patches as mpatches

SAVE_DIR = FIGURES_DIR / 'F_peak_lag'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('Setup done. Saving to:', SAVE_DIR)

In [ ]:
print('Loading atmosphere...'); atm_raw = load_csvs('atmosphere')
print('Loading surface...');    srf_raw = load_csvs('surface')
SURF_TEMP_COL, _ = detect_surface_cols(srf_raw)
atm = atm_raw[atm_raw[BUILDING_COL] == 0].copy() if BUILDING_COL in atm_raw.columns else atm_raw.copy()
srf = srf_raw.copy()
print('Loaded.')

## F1 — Dual-axis diurnal overlay: T_air vs T_surface (summer, per area)

In [ ]:
DATE = '15Aug'

fig, axes = plt.subplots(1, len(AREA_ORDER), figsize=(17, 5), constrained_layout=True)
fig.suptitle(f'Air vs Surface Temperature — {DATE_LABELS[DATE]} (peak lag illustration)',
             fontsize=13, fontweight='bold')

for ax, area in zip(axes, AREA_ORDER):
    color = AREA_COLORS[area]
    ax.set_title(AREA_LABELS[area], fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('T_air (°C)', color='steelblue')
    ax2 = ax.twinx()
    ax2.set_ylabel('T_surface (°C)', color='sienna')

    # Air
    sub_a = atm[(atm['area'] == area) & (atm['date'] == DATE)]
    if len(sub_a):
        hrs_a = sorted(sub_a['hour'].unique())
        mn_a  = sub_a.groupby('hour')[AIR_TEMP_COL].mean().reindex(hrs_a)
        ax.plot(hrs_a, mn_a, color='steelblue', lw=2.2, label='T_air')
        peak_a = int(mn_a.idxmax())
        ax.axvline(peak_a, color='steelblue', lw=1, ls=':', alpha=0.8)
        ax.text(peak_a + 0.2, mn_a.max(), f'peak {peak_a}h', fontsize=8, color='steelblue')

    # Surface
    sub_s = srf[(srf['area'] == area) & (srf['date'] == DATE)]
    if len(sub_s):
        hrs_s = sorted(sub_s['hour'].unique())
        mn_s  = sub_s.groupby('hour')[SURF_TEMP_COL].mean().reindex(hrs_s)
        ax2.plot(hrs_s, mn_s, color='sienna', lw=2.2, ls='--', label='T_surface')
        peak_s = int(mn_s.idxmax())
        ax2.axvline(peak_s, color='sienna', lw=1, ls=':', alpha=0.8)
        ax2.text(peak_s + 0.2, mn_s.max(), f'peak {peak_s}h', fontsize=8, color='sienna')

    ax.set_xticks(range(0, 24, 3))
    ax.tick_params(axis='y', labelcolor='steelblue')
    ax2.tick_params(axis='y', labelcolor='sienna')
    ax.grid(lw=0.4, alpha=0.4)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='lower right')

fig.savefig(SAVE_DIR / 'F1_dual_axis_summer.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved F1')

## F2 — Peak lag heatmap (surface peak − air peak) for all areas × seasons

In [ ]:
lag_mat = np.full((len(AREA_ORDER), len(DATE_ORDER)), np.nan)
air_peak_mat  = np.full_like(lag_mat, np.nan)
surf_peak_mat = np.full_like(lag_mat, np.nan)

for r, area in enumerate(AREA_ORDER):
    for c, date in enumerate(DATE_ORDER):
        sub_a = atm[(atm['area'] == area) & (atm['date'] == date)]
        sub_s = srf[(srf['area'] == area) & (srf['date'] == date)]
        if len(sub_a) == 0 or len(sub_s) == 0: continue
        peak_a = sub_a.groupby('hour')[AIR_TEMP_COL].mean().idxmax()
        peak_s = sub_s.groupby('hour')[SURF_TEMP_COL].mean().idxmax()
        air_peak_mat[r, c]  = peak_a
        surf_peak_mat[r, c] = peak_s
        lag_mat[r, c]       = peak_s - peak_a  # negative = surface peaks earlier

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
fig.suptitle('Peak Hour Analysis: Surface vs Air Temperature', fontsize=13, fontweight='bold')

for ax, mat, title, cmap, fmt in zip(
        axes,
        [air_peak_mat, surf_peak_mat, lag_mat],
        ['Air peak hour', 'Surface peak hour', 'Lag (surf − air) hours'],
        ['Blues', 'Oranges', 'RdBu_r'],
        ['{:.0f}h', '{:.0f}h', '{:+.0f}h']):

    vabs = np.nanmax(np.abs(mat)) if 'Lag' in title else None
    vmin = -vabs if vabs else None
    vmax =  vabs if vabs else None

    im = ax.imshow(mat, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(DATE_ORDER)))
    ax.set_xticklabels([DATE_LABELS[d].replace(' (', '\n(') for d in DATE_ORDER], fontsize=8)
    ax.set_yticks(range(len(AREA_ORDER)))
    ax.set_yticklabels([AREA_LABELS[a].replace('\n', ' ') for a in AREA_ORDER], fontsize=8)

    for i in range(len(AREA_ORDER)):
        for j in range(len(DATE_ORDER)):
            v = mat[i, j]
            if not np.isnan(v):
                ax.text(j, i, fmt.format(v), ha='center', va='center', fontsize=9,
                        color='white' if abs(v) > 0.6 * (np.nanmax(mat) or 1) else 'black')

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.savefig(SAVE_DIR / 'F2_peak_lag_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved F2')

## F3 — All seasons dual-axis overlay for a single area (Zefkseidos)

In [ ]:
FOCUS_AREA = 'Zefkseidos'

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
fig.suptitle(f'Air vs Surface Temperature — {AREA_LABELS[FOCUS_AREA].replace(chr(10)," ")} — All Seasons',
             fontsize=13, fontweight='bold')

for ax, date in zip(axes.ravel(), DATE_ORDER):
    color = DATE_COLORS[date]
    ax.set_title(DATE_LABELS[date], fontsize=10, fontweight='bold', color=color)
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('T_air (°C)', color='steelblue')
    ax2 = ax.twinx()
    ax2.set_ylabel('T_surface (°C)', color='sienna')

    sub_a = atm[(atm['area'] == FOCUS_AREA) & (atm['date'] == date)]
    sub_s = srf[(srf['area'] == FOCUS_AREA) & (srf['date'] == date)]

    if len(sub_a):
        hrs_a = sorted(sub_a['hour'].unique())
        mn_a  = sub_a.groupby('hour')[AIR_TEMP_COL].mean().reindex(hrs_a)
        ax.plot(hrs_a, mn_a, color='steelblue', lw=2, label='T_air')
        ax.fill_between(hrs_a, mn_a - sub_a.groupby('hour')[AIR_TEMP_COL].std().reindex(hrs_a),
                         mn_a + sub_a.groupby('hour')[AIR_TEMP_COL].std().reindex(hrs_a),
                         color='steelblue', alpha=0.12)
        peak_a = int(mn_a.idxmax())
        ax.axvline(peak_a, color='steelblue', lw=1, ls=':')

    if len(sub_s):
        hrs_s = sorted(sub_s['hour'].unique())
        mn_s  = sub_s.groupby('hour')[SURF_TEMP_COL].mean().reindex(hrs_s)
        ax2.plot(hrs_s, mn_s, color='sienna', lw=2, ls='--', label='T_surface')
        ax2.fill_between(hrs_s, mn_s - sub_s.groupby('hour')[SURF_TEMP_COL].std().reindex(hrs_s),
                          mn_s + sub_s.groupby('hour')[SURF_TEMP_COL].std().reindex(hrs_s),
                          color='sienna', alpha=0.12)
        peak_s = int(mn_s.idxmax())
        ax2.axvline(peak_s, color='sienna', lw=1, ls=':')
        lag = peak_s - peak_a if len(sub_a) else None
        if lag is not None:
            ax.text(0.98, 0.95, f'Lag: {lag:+d}h', transform=ax.transAxes,
                    ha='right', va='top', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='grey', alpha=0.85))

    ax.set_xticks(range(0, 24, 3))
    ax.tick_params(axis='y', labelcolor='steelblue', labelsize=8)
    ax2.tick_params(axis='y', labelcolor='sienna', labelsize=8)
    ax.grid(lw=0.4, alpha=0.4)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

fig.savefig(SAVE_DIR / f'F3_dual_axis_{FOCUS_AREA}_all_seasons.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved F3')

## F4 — Peak lag summary table

In [ ]:
rows = []
for area in AREA_ORDER:
    for date in DATE_ORDER:
        sub_a = atm[(atm['area'] == area) & (atm['date'] == date)]
        sub_s = srf[(srf['area'] == area) & (srf['date'] == date)]
        if len(sub_a) == 0 or len(sub_s) == 0: continue
        peak_a = int(sub_a.groupby('hour')[AIR_TEMP_COL].mean().idxmax())
        peak_s = int(sub_s.groupby('hour')[SURF_TEMP_COL].mean().idxmax())
        rows.append({'Area': area, 'Date': date,
                     'Air peak (h)': peak_a, 'Surface peak (h)': peak_s,
                     'Lag surf−air (h)': peak_s - peak_a})
lag_df = pd.DataFrame(rows)
print(lag_df.to_string(index=False))
print('\nAll done. Figures saved to:', SAVE_DIR)